In [ ]:
from groq import Groq

client = Groq(api_key="")

def get_propositions(text):
    """
    Step 1: Use the LLM to break text into atomic facts/propositions.
    """
    prompt = f"Break the following text into a list of standalone propositions (atomic facts). Return them as a bulleted list:\n\n{text}"
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    # Simple parsing of the bulleted list
    lines = response.choices[0].message.content.split('\n')
    return [line.strip('- ').strip() for line in lines if line.strip()]

def should_continue_chunk(current_chunk, next_proposition):
    """
    Step 2: The 'Agent' decides if the new fact fits the current context.
    """
    prompt = f"""
    Current Chunk: {current_chunk}
    Next Proposition: {next_proposition}
    
    Does the 'Next Proposition' continue the same specific topic or context as the 'Current Chunk'? 
    Answer only with 'YES' if it belongs together, or 'NO' if it introduces a new topic/context.
    """
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Using a smaller model for speed/cost
        messages=[{"role": "user", "content": prompt}]
    )
    return "YES" in response.choices[0].message.content.upper()

def agentic_chunker(text):
    propositions = get_propositions(text)
    chunks = []
    
    if not propositions:
        return chunks

    current_chunk = propositions[0]
    
    for i in range(1, len(propositions)):
        next_prop = propositions[i]
        
        # The Agent makes the call
        if should_continue_chunk(current_chunk, next_prop):
            current_chunk += " " + next_prop
        else:
            # Topic changed! Save the old chunk and start fresh
            chunks.append(current_chunk)
            current_chunk = next_prop
            
    # Don't forget the last one
    chunks.append(current_chunk)
    return chunks

# --- Example Usage ---
raw_text = """The exploration of Mars has shifted from simple flybys to sophisticated rovers like Perseverance,
which is currently searching for signs of ancient microbial life in Jezero Crater.
Scientists analyze soil samples to determine if the Red Planet once held liquid water,
a key ingredient for biological evolution. High-resolution cameras capture the desolate beauty of the Martian landscape,
sending data back to Earth via deep-space networks. Meanwhile, on our own planet,
the midnight zone of the ocean remains largely a mystery.
Gigantic squids and bioluminescent fish thrive in crushing pressures where sunlight never reaches.
Marine biologists use remotely operated vehicles to document these alien-like species,
revealing a complex ecosystem that relies on hydrothermal vents rather than photosynthesis.
Both environments present extreme challenges for robotic exploration and data transmission."""

final_chunks = agentic_chunker(raw_text)

for idx, chunk in enumerate(final_chunks):
    print(f"--- Chunk {idx+1} ---")
    print(chunk)

--- Chunk 1 ---
Here's the bulleted list of standalone propositions (atomic facts):
--- Chunk 2 ---
* The exploration of Mars has shifted from simple flybys to sophisticated rovers like Perseverance. * Perseverance is currently searching for signs of ancient microbial life in Jezero Crater. * Scientists analyze soil samples to determine if Mars once held liquid water. * Liquid water is a key ingredient for biological evolution. * High-resolution cameras capture the Martian landscape. * Data from Mars is sent back to Earth via deep-space networks.
--- Chunk 3 ---
* The midnight zone of the ocean remains largely a mystery. * Gigantic squids thrive in the midnight zone of the ocean. * Bioluminescent fish thrive in the midnight zone of the ocean. * The midnight zone of the ocean has crushing pressures. * Sunlight never reaches the midnight zone of the ocean. * Marine biologists use remotely operated vehicles to document species in the midnight zone. * The ecosystem in the midnight zone rel